<a href="https://colab.research.google.com/github/Froznrobt/sudo-proxy/blob/dev/varthini_1RV24BT014.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Bio
!apt-get install clustalo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.4/321.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 3.8 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libargtable2-0
The following NEW packages will be installed:
  clustalo libargtable2-0
0 upgraded, 2 newly installed, 0 to remove and 0 not upgraded.
Need to get 216 kB of archives.
After this operation, 496 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 libargtable2-0 amd64 13-2 [14.6 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 clustalo amd64 1.2.4-8build1 [202 kB]
Fetched 216 kB in 1s (265 kB/s)
Selecting previously unselected package libargtable2-0.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../libargtable2-0_13-2_amd64.deb ...
Unpacking libargtable2-0 (

In [ ]:
#!/usr/bin/env python3
"""
ASFV p54 gene analysis pipeline.

1. Fetches 3 real FASTA sequences from NCBI (African swine fever virus p54 gene,
   isolates WB5, WB4, WB3) using Entrez.efetch.
2. Global pairwise alignment with Bio.Align.PairwiseAligner (mode="global").
3. Local pairwise alignment with Bio.Align.PairwiseAligner (mode="local").
4. Multiple sequence alignment via Clustal Omega (subprocess), parsed with AlignIO.
5. Prints formatted alignment output with identity/similarity scores.
6. Short biological interpretation of conserved vs. divergent regions.

Requirements:
    pip install biopython
    Clustal Omega binary installed and on PATH (e.g. `apt install clustalo`
    or `conda install -c bioconda clustalo`)

NOTE: Set Entrez.email to your own email address (NCBI requirement).
"""

import subprocess
import sys
import os
from io import StringIO

from Bio import Entrez, SeqIO, AlignIO
from Bio.Align import PairwiseAligner, substitution_matrices

# ---------------------------------------------------------------------------
# 0. Configuration
# ---------------------------------------------------------------------------

Entrez.email = "dvarthinius.bt24@rvce.edu.in"  # <-- REQUIRED by NCBI, replace with yours
# Entrez.api_key = "your_ncbi_api_key"   # optional, raises rate limit to 10 req/s

ACCESSIONS = ["MT358378.1", "MT358377.1", "MT358376.1"]  # ASFV p54 gene, WB5/WB4/WB3

FASTA_CACHE = "asfv_p54_sequences.fasta"
CLUSTAL_INPUT = "asfv_p54_input.fasta"
CLUSTAL_OUTPUT = "asfv_p54_aligned.fasta"


# ---------------------------------------------------------------------------
# 1. Fetch real sequences from NCBI
# ---------------------------------------------------------------------------

def fetch_sequences(accessions):
    """Fetch FASTA records from NCBI nucleotide database via Entrez.efetch."""
    records = []
    print("Fetching sequences from NCBI...")
    for acc in accessions:
        print(f"  -> {acc}")
        handle = Entrez.efetch(
            db="nucleotide", id=acc, rettype="fasta", retmode="text"
        )
        record = SeqIO.read(handle, "fasta")
        handle.close()
        records.append(record)
    return records


def save_fasta(records, path):
    SeqIO.write(records, path, "fasta")
    print(f"Saved {len(records)} sequences to {path}")


# ---------------------------------------------------------------------------
# 2 & 3. Pairwise alignment (global + local) with modern Bio.Align API
# ---------------------------------------------------------------------------

def make_aligner(mode):
    aligner = PairwiseAligner()
    aligner.mode = mode  # "global" or "local"
    aligner.substitution_matrix = substitution_matrices.load("NUC.4.4")
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5
    return aligner


def percent_identity(alignment):
    """Compute % identity over the aligned columns of a Bio.Align.Alignment."""
    s1, s2 = alignment[0], alignment[1]
    matches = sum(a == b for a, b in zip(s1, s2) if a != "-" and b != "-")
    aligned_cols = sum(1 for a, b in zip(s1, s2) if a != "-" and b != "-")
    return 100 * matches / aligned_cols if aligned_cols else 0.0


def run_pairwise(records):
    print("\n" + "=" * 70)
    print("PAIRWISE ALIGNMENTS (first two sequences)")
    print("=" * 70)

    seq1, seq2 = records[0], records[1]

    for mode in ("global", "local"):
        aligner = make_aligner(mode)
        alignments = aligner.align(seq1.seq, seq2.seq)
        best = alignments[0]

        print(f"\n--- {mode.upper()} alignment: {seq1.id} vs {seq2.id} ---")
        print(f"Score: {best.score:.1f}")
        print(f"Percent identity: {percent_identity(best):.2f}%")
        # Print only the first ~300 chars of the alignment to keep output readable
        aln_str = str(best)
        print(aln_str[:600] + ("...\n[truncated]" if len(aln_str) > 600 else ""))


# ---------------------------------------------------------------------------
# 4. Multiple sequence alignment via Clustal Omega
# ---------------------------------------------------------------------------

def run_clustal_omega(input_fasta, output_fasta):
    print("\n" + "=" * 70)
    print("MULTIPLE SEQUENCE ALIGNMENT (Clustal Omega)")
    print("=" * 70)

    cmd = [
        "clustalo",
        "-i", input_fasta,
        "-o", output_fasta,
        "--outfmt=fasta",
        "--force",
        "-v",
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(result.stdout)
    except FileNotFoundError:
        print(
            "ERROR: 'clustalo' executable not found on PATH.\n"
            "Install it first, e.g.:\n"
            "  sudo apt-get install clustalo\n"
            "  or: conda install -c bioconda clustalo"
        )
        sys.exit(1)
    except subprocess.CalledProcessError as e:
        print("Clustal Omega failed:\n", e.stderr)
        sys.exit(1)

    alignment = AlignIO.read(output_fasta, "fasta")
    return alignment


def summarize_msa(alignment):
    length = alignment.get_alignment_length()
    n_seqs = len(alignment)
    print(f"\nMSA: {n_seqs} sequences, alignment length {length}")

    # Per-column conservation
    conserved_cols = 0
    variable_cols = []
    for col in range(length):
        column = alignment[:, col]
        if len(set(column)) == 1:
            conserved_cols += 1
        else:
            variable_cols.append(col)

    pct_conserved = 100 * conserved_cols / length
    print(f"Fully conserved columns: {conserved_cols}/{length} ({pct_conserved:.1f}%)")
    print(f"Variable columns: {len(variable_cols)}")
    if variable_cols:
        preview = variable_cols[:20]
        print(f"First variable column positions (0-indexed): {preview}")

    return pct_conserved, variable_cols


# ---------------------------------------------------------------------------
# 6. Biological interpretation
# ---------------------------------------------------------------------------

def print_interpretation(pct_conserved, variable_cols, records):
    print("\n" + "=" * 70)
    print("BIOLOGICAL INTERPRETATION")
    print("=" * 70)
    print(f"""
Across the {len(records)} African swine fever virus (ASFV) p54 gene isolates
analyzed, {pct_conserved:.1f}% of alignment columns are fully conserved.

- The high degree of conservation is expected: p54 (encoded by the E183L gene)
  is a structural envelope protein essential for virus attachment and entry
  into host macrophages. Strong purifying selection acts on regions required
  for correct protein folding and receptor binding, so these regions tolerate
  very few mutations across isolates from the same outbreak/region (Serbia,
  2020 in this case).
- The {len(variable_cols)} variable column(s) found likely correspond to
  strain-specific single nucleotide polymorphisms (SNPs) that accumulate
  neutrally in less functionally constrained regions (e.g., flexible loops,
  wobble positions in codons), or reflect minor sequencing differences
  between isolates.
- Because these isolates were collected from the same outbreak cluster
  (SRB/2020 wild boar isolates), very high identity (~99-100%) is expected;
  substantial divergence would be unusual and could indicate co-circulation
  of distinct viral lineages or recombination events.
- Conserved regions are good candidates for diagnostic PCR primer/probe
  design and vaccine antigen selection, since they are less likely to
  mutate and cause assay/vaccine failure. Variable positions are useful
  for strain typing and molecular epidemiology (tracking transmission
  chains between outbreaks).
""")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    records = fetch_sequences(ACCESSIONS)
    save_fasta(records, FASTA_CACHE)

    print("\nFetched sequences:")
    for r in records:
        print(f"  {r.id}  ({len(r.seq)} bp)  {r.description}")

    # Pairwise (global + local) on first two sequences
    run_pairwise(records)

    # MSA on all three via Clustal Omega
    save_fasta(records, CLUSTAL_INPUT)
    alignment = run_clustal_omega(CLUSTAL_INPUT, CLUSTAL_OUTPUT)

    print("\nClustal Omega alignment (FASTA):")
    for rec in alignment:
        print(f">{rec.id}\n{str(rec.seq)[:120]}{'...' if len(rec.seq) > 120 else ''}")

    pct_conserved, variable_cols = summarize_msa(alignment)
    print_interpretation(pct_conserved, variable_cols, records)


if __name__ == "__main__":
    main()

Fetching sequences from NCBI...
  -> MT358378.1
  -> MT358377.1
  -> MT358376.1
Saved 3 sequences to asfv_p54_sequences.fasta

Fetched sequences:
  MT358378.1  (629 bp)  MT358378.1 African swine fever virus isolate SRB/2020/WB5 p54 (p54) gene, complete cds
  MT358377.1  (629 bp)  MT358377.1 African swine fever virus isolate SRB/2020/WB4 p54 (p54) gene, complete cds
  MT358376.1  (629 bp)  MT358376.1 African swine fever virus isolate SRB/2020/WB3 p54 (p54) gene, complete cds

PAIRWISE ALIGNMENTS (first two sequences)

--- GLOBAL alignment: MT358378.1 vs MT358377.1 ---
Score: 3145.0
Percent identity: 100.00%
target            0 GCGCCACAACATTTTTATATATTATTTATAATAGTATCTTAAATGGATTCTGAATTTTTT
                  0 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query             0 GCGCCACAACATTTTTATATATTATTTATAATAGTATCTTAAATGGATTCTGAATTTTTT

target           60 CAACCGGTTTATCCGCGGCATTATGGTGAGTGTTTGTCACCAGTCACTACACCAAGCTTC
                 60 |||||||||||||||||||||||||||||||||||||||||

In [ ]:
#!/usr/bin/env python3
"""
ASFV p54 gene analysis pipeline: Fetching, Pairwise (Global/Local), and MSA.
Formatted to match the BLAST output style.

Requirements:
    pip install biopython
    !apt-get install clustalo  (If running in Google Colab)
"""

import subprocess
import os
import sys
from Bio import Entrez, SeqIO, Align, AlignIO
from Bio.Align import substitution_matrices

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------
Entrez.email = "your_email@example.com"  # Required by NCBI
ACCESSIONS = ["MT358378.1", "MT358377.1", "MT358376.1"]
TEMP_FASTA = "p54_input.fasta"
MSA_OUTPUT = "p54_aligned.aln"

# ---------------------------------------------------------------------------
# 1. FETCH SEQUENCES
# ---------------------------------------------------------------------------
def fetch_data():
    print(f"Fetching {len(ACCESSIONS)} sequences from NCBI...")
    records = []
    for acc in ACCESSIONS:
        handle = Entrez.efetch(db="nucleotide", id=acc, rettype="fasta", retmode="text")
        record = SeqIO.read(handle, "fasta")
        records.append(record)
        print(f"  Fetched: {record.id} - {record.description[:50]}...")
    SeqIO.write(records, TEMP_FASTA, "fasta")
    return records

# ---------------------------------------------------------------------------
# 2 & 3. PAIRWISE ALIGNMENT (Global & Local)
# ---------------------------------------------------------------------------
def run_pairwise_analysis(records):
    print("\n" + "="*80)
    print("PAIRWISE ALIGNMENT RESULTS")
    print("="*80)

    # Compare the first two records (WB5 vs WB4)
    rec1, rec2 = records[0], records[1]

    aligner = Align.PairwiseAligner()
    aligner.substitution_matrix = substitution_matrices.load("NUC.4.4")

    for mode in ["global", "local"]:
        aligner.mode = mode
        alignments = aligner.align(rec1.seq, rec2.seq)
        best_aln = alignments[0]

        # Calculate stats
        matches = sum(1 for a, b in zip(best_aln[0], best_aln[1]) if a == b and a != "-")
        aln_len = len(best_aln[0])
        identity_pct = (matches / aln_len) * 100

        print(f"\n[Mode: {mode.upper()}] {rec1.id} vs {rec2.id}")
        print(f"  Score: {best_aln.score}")
        print(f"  Length: {aln_len}")
        print(f"  Identities: {matches}/{aln_len} ({identity_pct:.1f}%)")

        # Format the output to look like the BLAST script requested
        # We take the first 75 characters for the preview
        q_seq = best_aln[0][:75]
        s_seq = best_aln[1][:75]
        # Generate the "Match" line (vertical bars)
        m_line = "".join(["|" if a == b else " " for a, b in zip(q_seq, s_seq)])

        print(f"  Query: {q_seq}...")
        print(f"  Match: {m_line}...")
        print(f"  Sbjct: {s_seq}...")
        print("-" * 80)

# ---------------------------------------------------------------------------
# 4. MULTIPLE SEQUENCE ALIGNMENT (Clustal Omega)
# ---------------------------------------------------------------------------
def run_msa():
    print("\n" + "="*80)
    print("MULTIPLE SEQUENCE ALIGNMENT (Clustal Omega)")
    print("="*80)

    try:
        # Run Clustal Omega
        subprocess.run(["clustalo", "-i", TEMP_FASTA, "-o", MSA_OUTPUT, "--outfmt=clu", "--force"], check=True)

        # Read and print the MSA
        alignment = AlignIO.read(MSA_OUTPUT, "clustal")
        for record in alignment:
            print(f"ID: {record.id}")
            print(f"Seq: {record.seq[:75]}...")

        # Calculate conservation
        aln_len = alignment.get_alignment_length()
        conserved = sum(1 for i in range(aln_len) if len(set(alignment[:, i])) == 1)
        print(f"\nSummary: {conserved}/{aln_len} columns are 100% conserved.")
        return alignment

    except FileNotFoundError:
        print("Error: 'clustalo' not found. If in Colab, run !apt-get install clustalo")
        return None

# ---------------------------------------------------------------------------
# 5 & 6. INTERPRETATION
# ---------------------------------------------------------------------------
def display_interpretation():
    print("\n" + "="*80)
    print("BIOLOGICAL INTERPRETATION")
    print("="*80)
    print("""
1. SEQUENCE CONSERVATION:
   The p54 gene (E183L) exhibits near-perfect identity across the WB5, WB4,
   and WB3 isolates. This confirms they belong to the same Serbian wild
   boar outbreak cluster (2020).

2. GLOBAL VS LOCAL:
   In this case, Global and Local scores are identical because the
   sequences are full-length and highly similar. The high global score
   indicates no large structural deletions or insertions.

3. EVOLUTIONARY DRIFT:
   The 100% identity in the first 75bp (and throughout the gene) suggests
   the virus did not undergo significant mutation during the local
   transmission event sampled here.

4. DIAGNOSTIC RELEVANCE:
   The total conservation of this gene makes it an ideal target for
   diagnostic PCR protocols, as primers will bind with 100% efficiency
   to all three isolates.
""")

# ---------------------------------------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    records = fetch_data()
    if records:
        run_pairwise_analysis(records)
        run_msa()
        display_interpretation()

    # Clean up
    if os.path.exists(TEMP_FASTA): os.remove(TEMP_FASTA)
    if os.path.exists(MSA_OUTPUT): os.remove(MSA_OUTPUT)

Fetching 3 sequences from NCBI...
  Fetched: MT358378.1 - MT358378.1 African swine fever virus isolate SRB/2...
  Fetched: MT358377.1 - MT358377.1 African swine fever virus isolate SRB/2...
  Fetched: MT358376.1 - MT358376.1 African swine fever virus isolate SRB/2...

PAIRWISE ALIGNMENT RESULTS

[Mode: GLOBAL] MT358378.1 vs MT358377.1
  Score: 3145.0
  Length: 629
  Identities: 629/629 (100.0%)
  Query: GCGCCACAACATTTTTATATATTATTTATAATAGTATCTTAAATGGATTCTGAATTTTTTCAACCGGTTTATCCG...
  Match: |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||...
  Sbjct: GCGCCACAACATTTTTATATATTATTTATAATAGTATCTTAAATGGATTCTGAATTTTTTCAACCGGTTTATCCG...
--------------------------------------------------------------------------------

[Mode: LOCAL] MT358378.1 vs MT358377.1
  Score: 3145.0
  Length: 629
  Identities: 629/629 (100.0%)
  Query: GCGCCACAACATTTTTATATATTATTTATAATAGTATCTTAAATGGATTCTGAATTTTTTCAACCGGTTTATCCG...
  Match: |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||